In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.multioutput import ClassifierChain
from sklearn.model_selection import RandomizedSearchCV
from pathlib import Path
import mlflow
import pickle
import time

# ==========================================
# 1. SETUP PATH & MLFLOW
# ==========================================
root_path = Path.cwd().parent
mlflow.set_tracking_uri((root_path / "mlruns").as_uri())
mlflow.set_experiment("07_XGBoost_Hyperparameter_Tuning")

print("⏳ 1. Memuat Data Hasil Seleksi Fitur MFO...")
train_df = pd.read_csv(root_path / "Data/processed/train_selected_features.csv")

# Fitur yang terpilih dari output MFO
selected_features = ['TIPI1', 'TIPI2', 'TIPI3', 'TIPI4', 'TIPI5', 'TIPI6', 'TIPI7', 'TIPI8', 'TIPI9', 'TIPI10', 'VCL1', 'VCL4', 'VCL5', 'VCL6', 
                     'VCL7', 'VCL8', 'VCL9', 'VCL10', 'VCL12', 'VCL13', 'VCL14', 'VCL15', 'gender', 'engnat', 'age', 'religion', 'orientation',
                     'race', 'voted', 'married', 'familysize']

target_cols = ['risk_depression', 'risk_anxiety', 'risk_stress']

# Persiapan data training
X_train = train_df[selected_features]
Y_train = train_df[target_cols]

print(f"✓ Data shape: {X_train.shape}")
print(f"✓ Target columns: {target_cols}")
print(f"✓ Jumlah fitur: {len(selected_features)}")

with mlflow.start_run(run_name="XGBoost_MultiLabel_ClassifierChain_Tuning"):
    
    # ==========================================
    # 2. DEFINISI RUANG HYPERPARAMETER (OPTIMIZED)
    # ==========================================
    print("\n⏳ 2. Mendefinisikan Ruang Hyperparameter untuk XGBoost...")
    
    # Parameter grid yang dioptimalkan untuk performa dan efisiensi komputasi
    param_grid = {
        # Jumlah boosting rounds - lebih banyak = lebih baik (dengan learning rate lebih kecil)
        'estimator__n_estimators': [100, 200, 300],
        
        # Kedalaman pohon - balance antara complexity dan generalisasi
        'estimator__max_depth': [3, 5, 7],
        
        # Learning rate - kontrol kecepatan pembelajaran
        'estimator__learning_rate': [0.05, 0.1, 0.2],
        
        # Subsample - fraksi samples yang digunakan per tree
        'estimator__subsample': [0.7, 0.8, 1.0],
        
        # Colsample_bytree - fraksi fitur yang digunakan per tree
        'estimator__colsample_bytree': [0.7, 0.8, 1.0],
        
        # Gamma - minimum loss reduction untuk split
        'estimator__gamma': [0, 0.1, 0.2, 0.5],
        
        # Min child weight - minimum sum of instance weight
        'estimator__min_child_weight': [1, 3, 5]
    }

    print(f"✓ Total kombinasi hyperparameter: ~{np.prod([len(v) for v in param_grid.values()])} (akan di-sample)")
    
    # ==========================================
    # 3. INISIALISASI BASE ESTIMATOR & MULTI-LABEL STRATEGY
    # ==========================================
    print("\n⏳ 3. Inisialisasi XGBoost Base Estimator dan ClassifierChain...")
    
    # Base estimator dengan parameter dasar yang sudah terbukti baik
    # PENTING: objective='binary:logistic' harus selalu diset untuk classifier
    base_xgb = xgb.XGBClassifier(
        objective='binary:logistic',    # Binary classification objective - WAJIB
        eval_metric='logloss',          # Mode classifier
        use_label_encoder=False,        # Hindari warning deprecated
        random_state=42,                # Reproducibility
        n_jobs=-1                       # Parallel processing
    )
    
    # Wrapper untuk multi-label classification dengan dependency chain
    # order='random' untuk variasi dalam urutan prediksi label
    multi_target_xgb = ClassifierChain(
        base_xgb, 
        order='random', 
        random_state=42
    )
    
    # Tambahkan objective ke param_grid untuk memastikan tetap binary:logistic selama tuning
    param_grid['estimator__objective'] = ['binary:logistic']
    
    print("✓ Base Estimator: XGBClassifier dengan eval_metric='logloss'")
    print("✓ Multi-Label Strategy: ClassifierChain(order='random', random_state=42)")

    # ==========================================
    # 4. HYPERPARAMETER TUNING DENGAN RANDOMIZEDSEARCHCV
    # ==========================================
    print("\n⏳ 4. Memulai RandomizedSearchCV untuk Hyperparameter Optimization...")
    print("   (Menggunakan n_iter=20 untuk balance antara kualitas dan waktu komputasi)")
    
    tuning_start = time.time()
    
    random_search = RandomizedSearchCV(
        estimator=multi_target_xgb,
        param_distributions=param_grid,
        n_iter=20,                      # Jumlah kombinasi yang akan dicoba
        cv=3,                           # 3-fold cross-validation
        scoring='f1_macro',             # Metrik evaluasi
        verbose=1,
        random_state=42,
        n_jobs=-1                       # Parallel processing untuk CV
    )

    # Fit model dengan training data
    random_search.fit(X_train, Y_train)
    
    tuning_time = time.time() - tuning_start

    # Ambil hasil terbaik
    best_params = random_search.best_params_
    best_score = random_search.best_score_

    print(f"\n✅ Hyperparameter Tuning Selesai dalam {tuning_time/60:.2f} menit")
    print(f"✅ Parameter Terbaik Ditemukan:")
    for param, value in best_params.items():
        print(f"   {param}: {value}")
    print(f"✅ Best CV F1-Score (Macro Average): {best_score:.4f}")

    # ==========================================
    # 5. TRAINING MODEL FINAL DENGAN PARAMETER OPTIMAL
    # ==========================================
    print("\n⏳ 5. Melatih Model Final dengan Parameter Terbaik...")
    
    training_start = time.time()
    
    # Gunakan best estimator dari RandomizedSearchCV
    final_model = random_search.best_estimator_
    final_model.fit(X_train, Y_train)
    
    training_time = time.time() - training_start
    
    print(f"✓ Training selesai dalam {training_time:.2f} detik")
    print(f"✓ Model Type: {type(final_model).__name__}")

    # ==========================================
    # 6. PENYIMPANAN MODEL (PICKLE)
    # ==========================================
    print("\n⏳ 6. Menyimpan Model Final...")
    
    model_dir = root_path / "models"
    model_dir.mkdir(parents=True, exist_ok=True)
    
    model_path = model_dir / "xgboost_classifier_chain.pkl"
    with open(model_path, "wb") as f:
        pickle.dump(final_model, f)
    
    print(f"✓ Model disimpan ke: {model_path}")

    # ==========================================
    # 7. MLFLOW TRACKING & LOGGING
    # ==========================================
    print("\n⏳ 7. Logging ke MLflow...")
    
    # Log best hyperparameters
    for param, value in best_params.items():
        mlflow.log_param(param, value)
    
    # Log model metadata
    mlflow.log_param("multi_label_strategy", "ClassifierChain")
    mlflow.log_param("classifier_chain_order", "random")
    mlflow.log_param("cv_folds", 3)
    mlflow.log_param("tuning_method", "RandomizedSearchCV")
    mlflow.log_param("n_iter_random_search", 20)
    
    # Log feature information
    mlflow.log_param("num_features", len(selected_features))
    mlflow.log_param("num_target_labels", len(target_cols))
    mlflow.log_param("train_samples", X_train.shape[0])
    
    # Log performance metrics
    mlflow.log_metric("best_cv_f1_macro", best_score)
    mlflow.log_metric("tuning_time_minutes", tuning_time/60)
    mlflow.log_metric("training_time_seconds", training_time)
    
    # Log model artifacts
    mlflow.log_artifact(str(model_path))
    
    print("✓ MLflow tracking complete")

    # ==========================================
    # 8. SUMMARY
    # ==========================================
    print("\n" + "="*70)
    print("🎉 HYPERPARAMETER TUNING SELESAI")
    print("="*70)
    print(f"Model Architecture: XGBoost + ClassifierChain")
    print(f"Best F1-Score (CV): {best_score:.4f}")
    print(f"Total Tuning Time: {tuning_time/60:.2f} menit")
    print(f"Model Saved: {model_path}")
    print("="*70)

d:\Amikom\Semester 6\Project Data Mining\Project\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
d:\Amikom\Semester 6\Project Data Mining\Project\venv\lib\site-packages\mlflow\tracking\_tracking_service\utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)


⏳ 1. Memuat Data Hasil Seleksi Fitur MFO...
✓ Data shape: (8724, 31)
✓ Target columns: ['risk_depression', 'risk_anxiety', 'risk_stress']
✓ Jumlah fitur: 31

⏳ 2. Mendefinisikan Ruang Hyperparameter untuk XGBoost...
✓ Total kombinasi hyperparameter: ~2916 (akan di-sample)

⏳ 3. Inisialisasi XGBoost Base Estimator dan ClassifierChain...
✓ Base Estimator: XGBClassifier dengan eval_metric='logloss'
✓ Multi-Label Strategy: ClassifierChain(order='random', random_state=42)

⏳ 4. Memulai RandomizedSearchCV untuk Hyperparameter Optimization...
   (Menggunakan n_iter=20 untuk balance antara kualitas dan waktu komputasi)
Fitting 3 folds for each of 20 candidates, totalling 60 fits


d:\Amikom\Semester 6\Project Data Mining\Project\venv\lib\site-packages\xgboost\core.py:158: UserWarning: [21:31:07] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0015a694724fa8361-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)



✅ Hyperparameter Tuning Selesai dalam 0.49 menit
✅ Parameter Terbaik Ditemukan:
   estimator__subsample: 0.7
   estimator__objective: binary:logistic
   estimator__n_estimators: 100
   estimator__min_child_weight: 5
   estimator__max_depth: 3
   estimator__learning_rate: 0.2
   estimator__gamma: 0.5
   estimator__colsample_bytree: 1.0
✅ Best CV F1-Score (Macro Average): 0.4878

⏳ 5. Melatih Model Final dengan Parameter Terbaik...


d:\Amikom\Semester 6\Project Data Mining\Project\venv\lib\site-packages\xgboost\core.py:158: UserWarning: [21:31:07] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0015a694724fa8361-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


✓ Training selesai dalam 0.39 detik
✓ Model Type: ClassifierChain

⏳ 6. Menyimpan Model Final...
✓ Model disimpan ke: d:\Amikom\Semester 6\Project Data Mining\Project\models\multilabel_xgboost_classifier_chain.pkl

⏳ 7. Logging ke MLflow...
✓ MLflow tracking complete

🎉 HYPERPARAMETER TUNING SELESAI
Model Architecture: XGBoost + ClassifierChain
Best F1-Score (CV): 0.4878
Total Tuning Time: 0.49 menit
Model Saved: d:\Amikom\Semester 6\Project Data Mining\Project\models\multilabel_xgboost_classifier_chain.pkl


In [2]:
import pickle
from pathlib import Path

root_path = Path.cwd().parent
model_path = root_path / "models/multilabel_xgboost_classifier_chain.pkl"

with open(model_path, "rb") as f:
    loaded_model = pickle.load(f)

# Cek objective setiap estimator dalam chain
print("Cek objective semua estimator dalam chain:")
for i, est in enumerate(loaded_model.estimators_):
    obj        = getattr(est, 'objective', 'tidak ada')
    n_classes  = getattr(est, 'n_classes_', 'tidak ada')
    print(f"  Estimator {i}: objective={obj}, n_classes={n_classes}")

Cek objective semua estimator dalam chain:
  Estimator 0: objective=binary:logistic, n_classes=2
  Estimator 1: objective=binary:logistic, n_classes=2
  Estimator 2: objective=binary:logistic, n_classes=2
